# 1ZQ5 + NADPH + CSV-SMILES Docking (Modified Colab Version)

这个 notebook 是基于你原始的 **[PHM5013] Molecular Docking Scripts** 做的最小结构修改版，主要变化只有四处：

1. **Step 1**：继续做 1ZQ5 蛋白准备，但会自动从原始 PDB 里提取  
   - 原始共晶 ligand（供 box 定义）
   - 原始 NADPH（供 Step 1.5 回载）

2. **Step 1.5（新增）**：把单独的 **NADPH** 重新加回准备好的 APO 蛋白里，生成 `holo_protein_with_nadph.pdb`

3. **Step 2**：不再上传 SDF/MOL2，而是直接读取 `merge_ROCS0.918_EON0.9136.csv`  
   - 第一列作为 **SMILES**
   - 优先用 `Catalog_ID` / `Title` 作为 ligand 名称

4. **Step 4**：对 `holo_protein_with_nadph.pdb` 和 Step 2 的配体进行 docking，并且：
   - 生成 **带 partial charges 的 receptor / ligand PDBQT**
   - 对 PDBQT 进行 **charge validation**
   - 输出 **top 10 最终 best poses 的 merged SDF**

## 主要输出文件
- `/content/apo_protein_clean.pdb`
- `/content/holo_protein_with_nadph.pdb`
- `/content/reference_ligand_from_raw.pdb`
- `/content/*_ligands_clean.sdf`
- `/content/vina_docking/summary_results.csv`
- `/content/vina_docking/charge_validation.csv`
- `/content/vina_docking/top_10_final_docked_poses.sdf`

## 说明
- 原 notebook 里的 **Step 3a / Step 3b** 我没有改。
- 这里的最终 merged SDF 是 **按每个 ligand 的 rank-1 pose 排序后保留 overall top 10**。


In [ ]:
# =============================================================================
#  STEP 1: Protein Preparation (modified for 1ZQ5 / ligand removal / NADPH removal)
# =============================================================================

# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

structure_source = "Fetch from RCSB PDB"  # @param ["Upload PDB file", "Fetch from RCSB PDB"]
pdb_id = "1ZQ5"  # @param {type:"string"}
target_pH = 7.4  # @param {type:"number"}
restraint_k = 500  # @param {type:"number"}
convergence_tol = 10  # @param {type:"number"}
max_iterations = 1  # @param {type:"integer"}

# NEW: names used to identify the cofactor to exclude from Step 1 and re-add in Step 1.5
cofactor_resnames_csv = "NAP,NAD,NADP,NDP,NPH,NADH"  # @param {type:"string"}


# ---------------------------------------------------------------------------
#  STEP 0 : Install Dependencies
# ---------------------------------------------------------------------------

import subprocess, sys

print("=" * 72)
print("  STEP 0 | Installing Dependencies")
print("=" * 72)

_packages = {
    "openmm":    "openmm",
    "pdbfixer":  "pdbfixer",
    "biopython": "Bio",
    "propka":    "propka",
    "pdb2pqr":   "pdb2pqr",
    "requests":  "requests",
    "numpy":     "numpy",
}

for pkg_name, import_name in _packages.items():
    try:
        __import__(import_name)
        print(f"  [OK]  {pkg_name:24s} already installed")
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pkg_name],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        print(f"  [OK]  {pkg_name:24s} installed")

print("  All dependencies ready.\n")


# ---------------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------------

import os, io, warnings, tempfile, shutil, re
import numpy as np
import requests

from pathlib import Path
from pdbfixer import PDBFixer
from openmm.app import (
    PDBFile, ForceField, Simulation,
)
from openmm import (
    LangevinMiddleIntegrator, CustomExternalForce, unit,
)
from Bio.PDB import PDBParser, PDBIO, Select

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------------
#  Helper Utilities
# ---------------------------------------------------------------------------

INTERMEDIATE_DIR = Path("/content/prep_intermediates")
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

COFACTOR_NAMES = {x.strip().upper() for x in cofactor_resnames_csv.split(",") if x.strip()}
COMMON_NONLIGAND = {
    "HOH","WAT","DOD","SO4","PO4","PEG","GOL","EDO","ACT","ACY","FMT",
    "CL","NA","K","MG","MN","CA","ZN","FE","CU","CD","IOD","BR"
}

def _print_header(step_num, title):
    print("\n" + "=" * 72)
    print(f"  STEP {step_num} | {title}")
    print("=" * 72)

def _residue_key_from_pdb_line(line):
    return (
        line[17:20].strip().upper(),
        line[21].strip() or "A",
        int(line[22:26]),
        line[26].strip() or " ",
    )

def _extract_het_groups(raw_pdb_path):
    groups = {}
    with open(raw_pdb_path) as fh:
        for line in fh:
            if not line.startswith("HETATM"):
                continue
            resn, chain, rnum, icode = _residue_key_from_pdb_line(line)
            if resn in {"HOH", "WAT", "DOD"}:
                continue
            key = (resn, chain, rnum, icode)
            groups.setdefault(key, []).append(line)
    return groups

def _write_group_to_pdb(lines, out_path, remark):
    with open(out_path, "w") as fh:
        fh.write(f"REMARK {remark}\n")
        for ln in lines:
            fh.write(ln if ln.endswith("\n") else ln + "\n")
        fh.write("END\n")

def _choose_reference_ligand(groups):
    candidates = []
    for key, lines in groups.items():
        resn = key[0]
        if resn in COFACTOR_NAMES or resn in COMMON_NONLIGAND:
            continue
        candidates.append((len(lines), key, lines))
    if not candidates:
        return None
    candidates.sort(reverse=True, key=lambda x: x[0])  # largest HETATM residue first
    return candidates[0]

def _atom_count(pdb_path):
    count = 0
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                count += 1
    return count


# ---------------------------------------------------------------------------
#  STEP 1 : Protein Input
# ---------------------------------------------------------------------------

_print_header(1, "Protein Input")

if structure_source == "Upload PDB file":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded. Please re-run and select a PDB file.")
    upload_name = list(uploaded.keys())[0]
    raw_path = str(INTERMEDIATE_DIR / "01_raw_input.pdb")
    with open(raw_path, "wb") as f:
        f.write(uploaded[upload_name])
    print(f"  Uploaded '{upload_name}' -> {raw_path}")

elif structure_source == "Fetch from RCSB PDB":
    if not pdb_id.strip():
        raise ValueError("Please enter a PDB accession ID in the pdb_id parameter.")
    _id = pdb_id.strip().upper()
    url = f"https://files.rcsb.org/download/{_id}.pdb"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise RuntimeError(f"Could not fetch PDB {_id} (HTTP {resp.status_code}).")
    raw_path = str(INTERMEDIATE_DIR / "01_raw_input.pdb")
    with open(raw_path, "w") as f:
        f.write(resp.text)
    print(f"  Fetched {_id} from RCSB -> {raw_path}")

else:
    raise ValueError("Invalid structure_source selection.")

# NEW: Extract reference ligand and raw cofactor from the original 1ZQ5 file
_print_header("1A", "Extracting raw reference ligand and raw NADPH candidate(s)")

het_groups = _extract_het_groups(raw_path)
if not het_groups:
    print("  No non-water HETATM groups found in the raw PDB.")
else:
    print(f"  Non-water HET groups found: {len(het_groups)}")
    summary = []
    for key, lines in sorted(het_groups.items()):
        resn, chain, rnum, icode = key
        summary.append((resn, chain, rnum, len(lines)))
    for resn, chain, rnum, nat in summary[:20]:
        print(f"    {resn:>4s}  chain {chain}  res {rnum:<4d}  atoms {nat}")
    if len(summary) > 20:
        print(f"    ... plus {len(summary)-20} more group(s)")

    # save the first/large cofactor group
    cofactor_candidates = [(k, v) for k, v in het_groups.items() if k[0] in COFACTOR_NAMES]
    if cofactor_candidates:
        # keep the largest one
        cofactor_candidates.sort(key=lambda kv: len(kv[1]), reverse=True)
        cof_key, cof_lines = cofactor_candidates[0]
        raw_cofactor_path = "/content/nadph_from_raw_reference.pdb"
        _write_group_to_pdb(
            cof_lines, raw_cofactor_path,
            f"Raw cofactor extracted from input PDB: {cof_key}"
        )
        print(f"  Raw cofactor saved -> {raw_cofactor_path}")
    else:
        print("  No cofactor residue matching cofactor_resnames_csv was found in raw PDB.")

    ref_pick = _choose_reference_ligand(het_groups)
    if ref_pick is not None:
        _, ref_key, ref_lines = ref_pick
        reference_ligand_path = "/content/reference_ligand_from_raw.pdb"
        _write_group_to_pdb(
            ref_lines, reference_ligand_path,
            f"Reference ligand extracted from input PDB: {ref_key}"
        )
        print(f"  Reference ligand saved -> {reference_ligand_path}")
    else:
        print("  No non-cofactor ligand candidate found for automatic reference-box extraction.")


# ---------------------------------------------------------------------------
#  STEP 2 : Initial Structure Audit
# ---------------------------------------------------------------------------

_print_header(2, "Initial Structure Audit")

parser = PDBParser(QUIET=True)
structure = parser.get_structure("input", raw_path)

models      = list(structure.get_models())
chains      = [c.id for c in models[0].get_chains()]
residues    = [r for r in models[0].get_residues() if r.id[0] == " "]
hetatm_res  = [r for r in models[0].get_residues() if r.id[0] not in (" ", "W")]
waters      = [r for r in models[0].get_residues() if r.id[0] == "W"]
atoms       = list(models[0].get_atoms())
altloc_atoms = [a for a in atoms if a.get_altloc() not in (" ", "", "A")]
ins_code_res = [r for r in models[0].get_residues() if r.id[2].strip()]

print(f"  {'Models':<40s} {len(models)}")
print(f"  {'Chains':<40s} {chains}")
print(f"  {'Standard AA residues':<40s} {len(residues)}")
print(f"  {'HETATM residues':<40s} {len(hetatm_res)}")
print(f"  {'Water molecules':<40s} {len(waters)}")
print(f"  {'Total atoms':<40s} {len(atoms)}")
print(f"  {'Alt-conformation atoms':<40s} {len(altloc_atoms)}")
print(f"  {'Residues with insertion codes':<40s} {len(ins_code_res)}")


# ---------------------------------------------------------------------------
#  STEP 3 : Chain Selection
# ---------------------------------------------------------------------------

_print_header(3, "Chain Selection")

print(f"  Chains available: {chains}")

if len(chains) == 1:
    selected_chain = chains[0]
    print(f"  Single chain '{selected_chain}' -- no selection needed.")
else:
    selected_chain = chains
    print(f"  Multiple chains found. Keeping all: {selected_chain}")

class ChainSelect(Select):
    def __init__(self, chain_ids):
        self.chain_ids = chain_ids if isinstance(chain_ids, list) else [chain_ids]
    def accept_chain(self, chain):
        return chain.id in self.chain_ids

chain_out = str(INTERMEDIATE_DIR / "02_chain_selected.pdb")
io_pdb = PDBIO()
io_pdb.set_structure(structure)
io_pdb.save(chain_out, ChainSelect(selected_chain))
print(f"  Chain-selected structure -> {chain_out}")


# ---------------------------------------------------------------------------
#  STEP 4 : PDBFixer -- Gaps, Missing Heavy Atoms, APO Conversion
# ---------------------------------------------------------------------------

_print_header(4, "PDBFixer: Gaps, Missing Heavy Atoms, APO Conversion")

fixer = PDBFixer(filename=chain_out)

fixer.findMissingResidues()
n_gaps = sum(len(v) for v in fixer.missingResidues.values())
if n_gaps > 0:
    print(f"  Found {n_gaps} missing residue(s) in sequence gaps -- adding ...")
else:
    print("  No sequence gaps.")

fixer.findNonstandardResidues()
if fixer.nonstandardResidues:
    print(f"  Replacing {len(fixer.nonstandardResidues)} non-standard residue(s) ...")
    fixer.replaceNonstandardResidues()
else:
    print("  No non-standard residues.")

print("  Removing all heteroatoms and water (original ligand + NADPH + solvent) ...")
fixer.removeHeterogens(True)

fixer.findMissingAtoms()
n_missing = sum(len(v) for v in fixer.missingAtoms.values())
n_terminal = sum(len(v) for v in fixer.missingTerminals.values())
print(f"  Missing heavy atoms: {n_missing}  |  Missing terminal atoms: {n_terminal}")
fixer.addMissingAtoms()
print("  All missing heavy atoms added.")

apo_path = str(INTERMEDIATE_DIR / "03_pdbfixer_apo.pdb")
PDBFile.writeFile(fixer.topology, fixer.positions, open(apo_path, "w"))
print(f"  PDBFixer output -> {apo_path}")


# ---------------------------------------------------------------------------
#  STEP 5 : BioPython cleanup
# ---------------------------------------------------------------------------

_print_header(5, "BioPython: Alt-Conformers, Insertion Codes, Single Model")

structure = parser.get_structure("apo", apo_path)

n_altloc_fixed = 0
for atom in structure.get_atoms():
    if atom.get_altloc() not in (" ", ""):
        atom.set_altloc(" ")
        atom.set_occupancy(1.0)
        n_altloc_fixed += 1
print(f"  Normalised {n_altloc_fixed} altloc atoms -> blank (occupancy 1.0).")

n_renumbered = 0
for model in structure:
    for chain in model:
        new_id = 1
        for residue in list(chain.get_residues()):
            old_id = residue.id
            if old_id[1] != new_id or old_id[2] != " ":
                residue.id = (old_id[0], new_id, " ")
                n_renumbered += 1
            new_id += 1

if n_renumbered > 0:
    print(f"  Renumbered {n_renumbered} residue(s) sequentially for pdb2pqr.")
else:
    print("  Residue numbering already sequential.")

cleaned_path = str(INTERMEDIATE_DIR / "04_biopy_cleaned.pdb")
io_pdb.set_structure(structure)
io_pdb.save(cleaned_path)
print(f"  Cleaned structure -> {cleaned_path}")


# ---------------------------------------------------------------------------
#  STEP 6 : PropKa
# ---------------------------------------------------------------------------

_print_header(6, "PropKa: Per-Residue pKa Prediction")

propka_report_path = str(INTERMEDIATE_DIR / "05_propka_report.pka")

try:
    from propka.run import single as propka_single
    propka_mol = propka_single(cleaned_path, optargs=["--pH", str(target_pH)])
    with open(propka_report_path, "w") as pka_out:
        propka_mol.write_pka(pka_out)
    print(f"  PropKa report -> {propka_report_path}")
except Exception as e:
    print(f"  WARNING: PropKa standalone run failed ({e}).")
    print("  pdb2pqr will run PropKa internally in Step 7.")


# ---------------------------------------------------------------------------
#  STEP 7 : pdb2pqr
# ---------------------------------------------------------------------------

_print_header(7, "pdb2pqr: Protonation + H-Bond Network Optimisation")

pqr_out   = str(INTERMEDIATE_DIR / "06_protonated_hbond.pqr")
pdb_h_out = str(INTERMEDIATE_DIR / "06_protonated_hbond.pdb")

pdb2pqr_cmd = [
    sys.executable, "-m", "pdb2pqr",
    "--ff", "AMBER",
    "--ffout", "AMBER",
    "--with-ph", str(target_pH),
    "--titration-state-method", "propka",
    "--pdb-output", pdb_h_out,
    cleaned_path,
    pqr_out,
]
result = subprocess.run(pdb2pqr_cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("  WARNING: pdb2pqr debumper failed -- retrying with --nodebump ...")
    pdb2pqr_cmd_nodebump = [
        sys.executable, "-m", "pdb2pqr",
        "--ff", "AMBER",
        "--ffout", "AMBER",
        "--with-ph", str(target_pH),
        "--titration-state-method", "propka",
        "--nodebump",
        "--pdb-output", pdb_h_out,
        cleaned_path,
        pqr_out,
    ]
    result = subprocess.run(pdb2pqr_cmd_nodebump, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("pdb2pqr failed.")

print("  pdb2pqr completed successfully.")

def _strip_resnames_from_pdb(in_pdb, out_pdb, banned_resnames={"HOH","WAT","DOD"}):
    removed = 0
    with open(in_pdb) as fin, open(out_pdb, "w") as fout:
        for line in fin:
            if line.startswith(("ATOM", "HETATM")):
                resn = line[17:20].strip().upper()
                if resn in banned_resnames:
                    removed += 1
                    continue
            fout.write(line)
    return removed

pdb_h_nowat_out = str(INTERMEDIATE_DIR / "06_protonated_hbond_nowat.pdb")
n_removed_water_lines = _strip_resnames_from_pdb(pdb_h_out, pdb_h_nowat_out)
print(f"  Removed {n_removed_water_lines} water ATOM/HETATM line(s) from pdb2pqr PDB before OpenMM.")

# Save unminimised protonated structure actually used for minimisation/fallback
unmin_path = str(INTERMEDIATE_DIR / "07_unminimised.pdb")
shutil.copy2(pdb_h_nowat_out, unmin_path)
print(f"  Unminimised protonated structure -> {unmin_path}")


# ---------------------------------------------------------------------------
#  STEP 8 : OpenMM minimisation
# ---------------------------------------------------------------------------

_print_header(8, "OpenMM: Restrained Energy Minimisation (AMBER14 + GBN2)")

final_path = "/content/apo_protein_clean.pdb"
min_path = str(INTERMEDIATE_DIR / "08_minimised.pdb")

try:
    pdb_in = PDBFile(pdb_h_nowat_out)
    ff = ForceField("amber14-all.xml", "implicit/gbn2.xml")
    system = ff.createSystem(pdb_in.topology, nonbondedCutoff=1.0 * unit.nanometers)

    restraint_force = CustomExternalForce(
        "0.5 * k * ((x - x0)^2 + (y - y0)^2 + (z - z0)^2)"
    )
    restraint_force.addGlobalParameter("k", restraint_k * unit.kilojoules_per_mole / unit.nanometers**2)
    restraint_force.addPerParticleParameter("x0")
    restraint_force.addPerParticleParameter("y0")
    restraint_force.addPerParticleParameter("z0")

    for i, atom in enumerate(pdb_in.topology.atoms()):
        if atom.element.symbol != "H":
            pos = pdb_in.positions[i]
            restraint_force.addParticle(i, [pos.x, pos.y, pos.z])

    system.addForce(restraint_force)

    integrator = LangevinMiddleIntegrator(
        300 * unit.kelvin, 1.0 / unit.picoseconds, 0.002 * unit.picoseconds
    )
    simulation = Simulation(pdb_in.topology, system, integrator)
    simulation.context.setPositions(pdb_in.positions)
    simulation.minimizeEnergy(
        tolerance=convergence_tol * unit.kilojoules_per_mole / unit.nanometers,
        maxIterations=max_iterations,
    )

    state_after = simulation.context.getState(getEnergy=True, getPositions=True)
    PDBFile.writeFile(simulation.topology, state_after.getPositions(), open(min_path, "w"))
    print(f"  OpenMM minimisation succeeded -> {min_path}")

except Exception as e:
    print(f"  WARNING: OpenMM minimisation skipped due to: {e}")
    print("  Falling back to protonated no-water structure for downstream docking.")
    shutil.copy2(pdb_h_nowat_out, min_path)

shutil.copy2(min_path, final_path)
print(f"  Final APO output -> {final_path}")


# ---------------------------------------------------------------------------
#  STEP 9 : Final Validation
# ---------------------------------------------------------------------------

_print_header(9, "Final Validation and Docking Readiness Checklist")

struct_final = parser.get_structure("final", final_path)
model_final = list(struct_final.get_models())[0]
chains_final  = [c.id for c in model_final.get_chains()]
res_final     = [r for r in model_final.get_residues() if r.id[0] == " "]
het_final     = [r for r in model_final.get_residues() if r.id[0] not in (" ", "W")]
wat_final     = [r for r in model_final.get_residues() if r.id[0] == "W"]
atoms_final   = list(model_final.get_atoms())

print(f"  Chains: {chains_final}  |  Residues: {len(res_final)}  |  Atoms: {len(atoms_final)}")
print(f"  HET residues after Step 1: {len(het_final)} (should be 0 before Step 1.5)")
print(f"  Water after Step 1: {len(wat_final)} (should be 0)")
print(f"  APO protein ready -> {final_path}")

if os.path.exists("/content/reference_ligand_from_raw.pdb"):
    print("  Reference ligand available for Step 4 box definition -> /content/reference_ligand_from_raw.pdb")
if os.path.exists("/content/nadph_from_raw_reference.pdb"):
    print("  Raw NADPH reference extracted -> /content/nadph_from_raw_reference.pdb")

print("\n  Protein is READY for Step 1.5 (re-add NADPH) and Step 4 docking.")

In [ ]:
# =============================================================================
#  STEP 1.5: Load NADPH and merge into prepared protein (NEW)
# =============================================================================

# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

nadph_source = "Use extracted NADPH from Step 1 if available"  # @param ["Use extracted NADPH from Step 1 if available", "Upload NADPH file"]
nadph_input_format = "pdb"  # @param ["pdb", "mol2"]


# ---------------------------------------------------------------------------
#  STEP 0 : Install Dependency
# ---------------------------------------------------------------------------

import sys, subprocess, os, shutil
from pathlib import Path

print("=" * 72)
print("  STEP 0 | Installing Dependencies for Step 1.5")
print("=" * 72)

try:
    from openbabel import openbabel as ob
    print("  [OK]  openbabel already installed")
except ImportError:
    subprocess.check_call(["apt-get", "install", "-y", "-q", "openbabel"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openbabel-wheel"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    from openbabel import openbabel as ob
    print("  [OK]  openbabel installed")


# ---------------------------------------------------------------------------
#  Helper
# ---------------------------------------------------------------------------

def _convert_mol2_to_pdb(mol2_path, pdb_path):
    obc = ob.OBConversion()
    obc.SetInAndOutFormats("mol2", "pdb")
    mol = ob.OBMol()
    if not obc.ReadFile(mol, str(mol2_path)):
        raise RuntimeError(f"Could not read MOL2: {mol2_path}")
    if not obc.WriteFile(mol, str(pdb_path)):
        raise RuntimeError(f"Could not write PDB: {pdb_path}")

def _clean_pdb_lines_for_merge(pdb_path, keep_hetatm=True):
    lines = []
    with open(pdb_path) as fh:
        for line in fh:
            if line.startswith("ATOM"):
                lines.append(line if line.endswith("\n") else line + "\n")
            elif keep_hetatm and line.startswith("HETATM"):
                lines.append(line if line.endswith("\n") else line + "\n")
    return lines


# ---------------------------------------------------------------------------
#  STEP 1 : Resolve inputs
# ---------------------------------------------------------------------------

print("\n" + "=" * 72)
print("  STEP 1 | Resolving Protein + NADPH Inputs")
print("=" * 72)

protein_pdb = Path("/content/apo_protein_clean.pdb")
if not protein_pdb.exists():
    raise FileNotFoundError("Expected /content/apo_protein_clean.pdb from Step 1, but it was not found.")

nadph_pdb = None

if nadph_source == "Use extracted NADPH from Step 1 if available" and Path("/content/nadph_from_raw_reference.pdb").exists():
    nadph_pdb = Path("/content/nadph_from_raw_reference.pdb")
    print(f"  Using extracted raw NADPH -> {nadph_pdb}")
else:
    from google.colab import files
    print("  Upload NADPH file (.pdb or .mol2):")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No NADPH file uploaded.")
    up_name = list(uploaded.keys())[0]
    up_path = Path(up_name)
    if nadph_input_format == "pdb":
        nadph_pdb = Path("/content/uploaded_nadph.pdb")
        shutil.copy2(up_path, nadph_pdb)
    else:
        nadph_pdb = Path("/content/uploaded_nadph_from_mol2.pdb")
        _convert_mol2_to_pdb(up_path, nadph_pdb)
    print(f"  NADPH prepared -> {nadph_pdb}")


# ---------------------------------------------------------------------------
#  STEP 2 : Merge APO protein + NADPH
# ---------------------------------------------------------------------------

print("\n" + "=" * 72)
print("  STEP 2 | Merging APO protein with NADPH")
print("=" * 72)

prot_lines = _clean_pdb_lines_for_merge(protein_pdb, keep_hetatm=False)
cof_lines  = _clean_pdb_lines_for_merge(nadph_pdb, keep_hetatm=True)

if not prot_lines:
    raise RuntimeError("No ATOM records found in apo protein PDB.")
if not cof_lines:
    raise RuntimeError("No HETATM/ATOM records found in NADPH PDB.")

merged_pdb = Path("/content/holo_protein_with_nadph.pdb")
with open(merged_pdb, "w") as fh:
    fh.write("REMARK APO protein from Step 1 + NADPH added back in Step 1.5\n")
    for ln in prot_lines:
        fh.write(ln)
    fh.write("TER\n")
    for ln in cof_lines:
        fh.write(ln)
    fh.write("END\n")

print(f"  Holo receptor saved -> {merged_pdb}")
print(f"  Protein atoms : {sum(1 for x in prot_lines if x.startswith('ATOM'))}")
print(f"  NADPH atoms   : {sum(1 for x in cof_lines if x.startswith(('ATOM','HETATM')))}")


# ---------------------------------------------------------------------------
#  STEP 3 : Quick validation
# ---------------------------------------------------------------------------

print("\n" + "=" * 72)
print("  STEP 3 | Quick Validation")
print("=" * 72)

print(f"  APO protein exists         : {protein_pdb.exists()}")
print(f"  NADPH file exists          : {nadph_pdb.exists()}")
print(f"  Final holo receptor exists : {merged_pdb.exists()}")
print("  Step 1.5 complete. Use /content/holo_protein_with_nadph.pdb in Step 4.")

In [ ]:
# =============================================================================
#  STEP 2: Ligand Preparation from CSV SMILES (modified)
# =============================================================================

# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

output_filename = "merge_ROCS0_918_EON0_9136_ligands"  # @param {type:"string"}
protonation_method = "OpenBabel (pH-aware)"  # @param ["OpenBabel (pH-aware)", "RDKit (keep formal charges as-is)"]
target_pH = 7.4  # @param {type:"number"}
n_conformers = 1  # @param {type:"integer"}
mmff_max_iters = 2000  # @param {type:"integer"}
max_ligands_to_prepare = 0  # @param {type:"integer"}
prefer_id_columns_csv = "Catalog_ID,Title,VIDA ID"  # @param {type:"string"}


# ---------------------------------------------------------------------------
#  Resolve parameter values
# ---------------------------------------------------------------------------

OUTPUT_BASENAME    = output_filename.strip().replace(" ", "_") + "_clean"
PROTONATION_METHOD = "openbabel" if "OpenBabel" in protonation_method else "rdkit_keep"
TARGET_PH          = target_pH
N_CONFORMERS       = n_conformers
MMFF_MAX_ITERS     = mmff_max_iters
MAX_LIGANDS        = max_ligands_to_prepare if max_ligands_to_prepare > 0 else None
PREFERRED_ID_COLS  = [x.strip() for x in prefer_id_columns_csv.split(",") if x.strip()]


# ---------------------------------------------------------------------------
#  STEP 0 : Install Dependencies
# ---------------------------------------------------------------------------

import sys, subprocess

def _print_header(step_num, title):
    print("\n" + "=" * 72)
    print(f"  STEP {step_num} | {title}")
    print("=" * 72)

print("=" * 72)
print("  STEP 0 | Installing Dependencies")
print("=" * 72)

def _run(cmd):
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for label, pip_name, import_name in [
    ("rdkit",     "rdkit",     "rdkit.Chem"),
    ("tqdm",      "tqdm",      "tqdm"),
    ("pandas",    "pandas",    "pandas"),
]:
    try:
        __import__(import_name)
        print(f"  [OK]  {label:24s} already installed")
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        print(f"  [OK]  {label:24s} installed")

try:
    from openbabel import openbabel as ob
    print(f"  [OK]  {'openbabel':24s} already installed")
except ImportError:
    _run(["apt-get", "install", "-y", "-q", "openbabel"])
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openbabel-wheel"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    from openbabel import openbabel as ob
    print(f"  [OK]  {'openbabel':24s} installed")

print("  All dependencies ready.\n")


# ---------------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------------

import os, time, warnings, tempfile, re
from copy import deepcopy
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdDistGeom
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import SDWriter

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

print("  Configuration")
print("  " + "-" * 60)
print(f"  Output basename  : {OUTPUT_BASENAME}")
print(f"  Protonation      : {PROTONATION_METHOD.upper()}")
print(f"  Target pH        : {TARGET_PH}")
print(f"  Conformers       : {N_CONFORMERS}")
print(f"  MMFF94s max iter : {MMFF_MAX_ITERS}")
print(f"  Ligand cap       : {MAX_LIGANDS if MAX_LIGANDS else 'ALL'}")


# ---------------------------------------------------------------------------
#  STEP 1 : File Upload
# ---------------------------------------------------------------------------

_print_header(1, "CSV Upload")

from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file uploaded. Please re-run the cell.")

upload_name = list(uploaded.keys())[0]
if not upload_name.lower().endswith(".csv"):
    raise ValueError("Please upload a CSV file. Expected merge_ROCS0.918_EON0.9136.csv")

csv_path = Path(upload_name)
print(f"  Uploaded : {upload_name}")


# ---------------------------------------------------------------------------
#  STEP 2 : Read CSV and build ligand records
# ---------------------------------------------------------------------------

_print_header(2, "Read CSV and Extract SMILES")

df = pd.read_csv(csv_path)
if df.shape[1] < 1:
    raise ValueError("CSV has no columns.")

smiles_col = df.columns[0]
print(f"  Using first column as SMILES -> {smiles_col}")

id_col = None
for col in PREFERRED_ID_COLS:
    if col in df.columns:
        id_col = col
        break

if id_col is None:
    print("  No preferred ID column found; ligand names will use row indices.")
else:
    print(f"  Using ligand identifier column -> {id_col}")

records = []
for i, row in df.iterrows():
    smi = str(row[smiles_col]).strip()
    if not smi or smi.lower() == "nan":
        continue
    lig_name = str(row[id_col]).strip() if id_col else f"LIG_{i+1:04d}"
    if not lig_name or lig_name.lower() == "nan":
        lig_name = f"LIG_{i+1:04d}"
    lig_name = re.sub(r"[^\w\-]", "_", lig_name)
    records.append({"name": lig_name, "smiles": smi, "row_idx": i+1})

if MAX_LIGANDS is not None:
    records = records[:MAX_LIGANDS]

print(f"  Valid ligand records: {len(records)}")
if len(records) == 0:
    raise ValueError("No valid SMILES were found in the first CSV column.")


# ---------------------------------------------------------------------------
#  STEP 3 : Ligand Preparation
# ---------------------------------------------------------------------------

_print_header(3, "Ligand Preparation from SMILES")

def standardise_mol(mol):
    lfc = rdMolStandardize.LargestFragmentChooser()
    te  = rdMolStandardize.TautomerEnumerator()
    mol = lfc.choose(mol)
    mol = te.Canonicalize(mol)
    return mol

def protonate_openbabel_from_smiles(smiles, ph=7.4):
    obconv = ob.OBConversion()
    obconv.SetInAndOutFormats("smi", "smi")
    obmol = ob.OBMol()
    if not obconv.ReadString(obmol, smiles):
        raise RuntimeError("OpenBabel failed to read SMILES.")
    obmol.AddHydrogens(False, True, ph)
    out_smi = obconv.WriteString(obmol).strip()
    return out_smi

def embed_and_minimise(mol, n_confs, max_iters):
    params = rdDistGeom.ETKDGv3()
    params.randomSeed            = 42
    params.numThreads            = 0
    params.useSmallRingTorsions  = True
    params.useMacrocycleTorsions = True
    params.enforceChirality      = True

    mol3d = Chem.AddHs(mol, addCoords=False)
    conf_ids = AllChem.EmbedMultipleConfs(mol3d, numConfs=n_confs, params=params)
    if not conf_ids:
        AllChem.EmbedMolecule(mol3d, AllChem.ETKDG())
    results = AllChem.MMFFOptimizeMoleculeConfs(
        mol3d, mmffVariant="MMFF94s", maxIters=max_iters
    )
    if not results:
        return mol3d, False
    energies = [(r[1], i) for i, r in enumerate(results) if r[0] == 0]
    if not energies:
        return mol3d, False
    _, best_conf = min(energies)
    mol_out = deepcopy(mol3d)
    for cid in reversed(range(mol_out.GetNumConformers())):
        if cid != best_conf:
            mol_out.RemoveConformer(cid)
    return mol_out, True

prepared = []
failed = []

for rec in tqdm(records, desc="  Preparing", unit="lig"):
    lig_name = rec["name"]
    smi = rec["smiles"]
    try:
        if PROTONATION_METHOD == "openbabel":
            smi_use = protonate_openbabel_from_smiles(smi, ph=TARGET_PH)
        else:
            smi_use = smi

        mol = Chem.MolFromSmiles(smi_use)
        if mol is None:
            raise RuntimeError("RDKit could not parse SMILES.")

        mol = standardise_mol(mol)
        mol.SetProp("_Name", lig_name)
        mol.SetProp("Original_SMILES", smi)

        mol3d, converged = embed_and_minimise(mol, n_confs=N_CONFORMERS, max_iters=MMFF_MAX_ITERS)
        if mol3d.GetNumConformers() == 0:
            raise RuntimeError("3D conformer generation failed.")

        mol3d.SetProp("_Name", lig_name)
        mol3d.SetProp("Converged", str(converged))
        mol3d.SetProp("Original_SMILES", smi)
        prepared.append((mol3d, lig_name))
    except Exception as e:
        failed.append((lig_name, str(e)))

print(f"  Prepared successfully : {len(prepared)}")
print(f"  Failed                : {len(failed)}")
if failed[:10]:
    print("  First failures:")
    for name, why in failed[:10]:
        print(f"    {name}: {why}")


# ---------------------------------------------------------------------------
#  STEP 4 : Write output SDF
# ---------------------------------------------------------------------------

_print_header(4, "Writing Output SDF")

out_sdf = Path(f"/content/{OUTPUT_BASENAME}.sdf")
with SDWriter(str(out_sdf)) as w:
    w.SetKekulize(False)
    for mol, name in prepared:
        w.write(mol)

prep_summary_csv = Path(f"/content/{OUTPUT_BASENAME}_prep_summary.csv")
with open(prep_summary_csv, "w") as fh:
    fh.write("Ligand_name,Status\n")
    for _, name in prepared:
        fh.write(f"{name},SUCCESS\n")
    for name, why in failed:
        fh.write(f"{name},FAILED: {str(why).replace(',', ';')}\n")

print(f"  Output SDF         -> {out_sdf}")
print(f"  Preparation report -> {prep_summary_csv}")

from google.colab import files as colab_files
colab_files.download(str(out_sdf))
colab_files.download(str(prep_summary_csv))

In [ ]:
# =============================================================================
#  STEP 4: Molecular Docking Run (modified for Step 1.5 receptor + partial charge validation)
# =============================================================================

# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

BOX_MODE = "D -- Reference ligand"  # @param ["A -- Blind docking", "B -- Manual centre + box", "C -- Residue-guided", "D -- Reference ligand"]

BLIND_PADDING = 5.0  # @param {type:"number"}

MANUAL_CENTER_X = 0.0   # @param {type:"number"}
MANUAL_CENTER_Y = 0.0   # @param {type:"number"}
MANUAL_CENTER_Z = 0.0   # @param {type:"number"}
MANUAL_SIZE_X   = 25.0  # @param {type:"number"}
MANUAL_SIZE_Y   = 25.0  # @param {type:"number"}
MANUAL_SIZE_Z   = 25.0  # @param {type:"number"}

KEY_RESIDUES_CSV = "54,55,117,118,308,310"  # @param {type:"string"}
RESIDUE_PADDING  = 8.0  # @param {type:"number"}

REF_LIGAND_PADDING = 6.0  # @param {type:"number"}

SCORING_FUNCTION = "vina"  # @param ["vina", "vinardo"]
EXHAUSTIVENESS   = 8       # @param {type:"integer"}
N_POSES          = 10      # @param {type:"integer"}
CPU_CORES        = 0       # @param {type:"integer"}
RANDOM_SEED      = 42      # @param {type:"integer"}

RUN_NAME          = "1ZQ5_NADPH_screening"  # @param {type:"string"}
AFFINITY_CUTOFF   = "none"  # @param {type:"string"}
SKIP_FAILED       = True    # @param {type:"boolean"}
LIGAND_LIMIT      = 0       # @param {type:"integer"}
FINAL_TOP_N_POSES = 10      # @param {type:"integer"}


# ---------------------------------------------------------------------------
#  Resolve parameters
# ---------------------------------------------------------------------------

import subprocess, sys, os, shutil, time, zipfile, csv, re, copy
from pathlib import Path

box_mode          = BOX_MODE[0]
scoring_fn        = SCORING_FUNCTION.strip().lower()
exhaustiveness    = EXHAUSTIVENESS
n_poses           = N_POSES
cpu_cores         = CPU_CORES
seed              = RANDOM_SEED
run_name          = re.sub(r"[^\w\-]", "_", RUN_NAME) if RUN_NAME else "run1"
affinity_cutoff   = None if AFFINITY_CUTOFF.strip().lower() in ("none", "") else float(AFFINITY_CUTOFF)
skip_failed       = SKIP_FAILED
ligand_limit      = LIGAND_LIMIT if LIGAND_LIMIT > 0 else None
final_top_n       = FINAL_TOP_N_POSES
key_res           = [int(r.strip()) for r in KEY_RESIDUES_CSV.split(",") if r.strip().isdigit()]
pad_auto          = BLIND_PADDING
pad_res           = RESIDUE_PADDING
pad_ref           = REF_LIGAND_PADDING
center            = None
size              = None
ref_lig_path      = None

if box_mode == "B":
    center = [MANUAL_CENTER_X, MANUAL_CENTER_Y, MANUAL_CENTER_Z]
    size   = [MANUAL_SIZE_X,   MANUAL_SIZE_Y,   MANUAL_SIZE_Z]

if box_mode == "C" and not key_res:
    print("  WARNING: No valid residues in KEY_RESIDUES_CSV -- falling back to blind docking (A)")
    box_mode = "A"


# ---------------------------------------------------------------------------
#  STEP 1 : Install Dependencies
# ---------------------------------------------------------------------------

def _pip(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    return r.returncode == 0

def _print_header(step_label, title):
    print("\n" + "=" * 72)
    print(f"  {step_label} | {title}")
    print("=" * 72)

_print_header("STEP 1/7", "Installing Dependencies")

_PKGS = {
    "vina"            : "AutoDock Vina 1.2.x Python bindings",
    "openbabel-wheel" : "Format conversion (PDB/SDF to PDBQT)",
}
for pkg, desc in _PKGS.items():
    status = "installed" if _pip(pkg) else "FAILED"
    print(f"  [{'OK' if status == 'installed' else '!!'}]  {pkg:<22s} {desc}  ({status})")

try:
    from rdkit import Chem
    print(f"  [OK]  {'rdkit':<22s} already installed")
except ImportError:
    _pip("rdkit-pypi")
    print(f"  [OK]  {'rdkit':<22s} installed")

print("  All dependencies ready.")


# ---------------------------------------------------------------------------
#  STEP 2 : Resolve files
# ---------------------------------------------------------------------------

import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Geometry import Point3D
from openbabel import openbabel as ob

WORKDIR = Path("/content/vina_docking")
WORKDIR.mkdir(exist_ok=True)

_print_header("STEP 2/7", "Resolving Input Files")

# receptor from Step 1.5
prot_in = Path("/content/holo_protein_with_nadph.pdb")
if not prot_in.exists():
    raise FileNotFoundError("Expected /content/holo_protein_with_nadph.pdb from Step 1.5, but it was not found.")

# ligands from Step 2
candidate_sdfs = sorted(Path("/content").glob("*_clean.sdf"))
lig_in = None
for p in candidate_sdfs:
    if "ligands_clean" in p.name.lower() or "merge_rocs" in p.name.lower():
        lig_in = p
        break
if lig_in is None and candidate_sdfs:
    lig_in = candidate_sdfs[-1]

if lig_in is None or not lig_in.exists():
    raise FileNotFoundError("No prepared ligand SDF from Step 2 was found in /content.")

# reference ligand from Step 1 (if using mode D)
if box_mode == "D":
    auto_ref = Path("/content/reference_ligand_from_raw.pdb")
    if auto_ref.exists():
        ref_lig_path = auto_ref
        print(f"  Using auto-extracted reference ligand -> {ref_lig_path}")
    else:
        from google.colab import files
        print("  Upload reference ligand (.sdf / .mol2 / .pdb):")
        ref_up = files.upload()
        ref_name = list(ref_up.keys())[0]
        ref_lig_path = WORKDIR / ref_name
        shutil.copy2(ref_name, ref_lig_path)

print(f"  Receptor : {prot_in}")
print(f"  Ligands  : {lig_in}")


# ---------------------------------------------------------------------------
#  STEP 3 : Compute box and receptor PDBQT
# ---------------------------------------------------------------------------

_print_header("STEP 3/7", "Box Definition + Receptor PDBQT")

# 3a. parse receptor coordinates
all_coords, res_coords, ca_coords = [], {}, []
with open(prot_in) as fh:
    for line in fh:
        if not line.startswith(("ATOM", "HETATM")):
            continue
        try:
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            rnum    = int(line[22:26])
            aname   = line[12:16].strip()
            all_coords.append([x, y, z])
            res_coords.setdefault(rnum, []).append([x, y, z])
            if aname == "CA" and line.startswith("ATOM"):
                ca_coords.append([x, y, z])
        except Exception:
            pass

if not all_coords:
    raise RuntimeError("No coordinates found in holo receptor PDB.")
coords_np = np.array(all_coords)

# 3b. compute docking box
if box_mode == "C":
    res_pts = []
    for r in key_res:
        if r in res_coords:
            res_pts.extend(res_coords[r])
    if res_pts:
        rp = np.array(res_pts)
        center = rp.mean(axis=0).tolist()
        size   = (rp.max(axis=0) - rp.min(axis=0) + 2 * pad_res).clip(min=15).tolist()
    else:
        print("  WARNING: residue-guided box failed; falling back to blind docking.")
        box_mode = "A"

if box_mode == "D" and ref_lig_path is not None:
    ext = ref_lig_path.suffix.lower()
    ref_mol = None
    if ext == ".sdf":
        ref_mol = next((m for m in Chem.SDMolSupplier(str(ref_lig_path), removeHs=False) if m is not None), None)
    elif ext == ".mol2":
        ref_mol = Chem.MolFromMol2File(str(ref_lig_path), removeHs=False)
    elif ext == ".pdb":
        ref_mol = Chem.MolFromPDBFile(str(ref_lig_path), removeHs=False)

    if ref_mol is None:
        raise RuntimeError("Could not parse reference ligand for BOX_MODE D.")
    ref_pos = np.array([ref_mol.GetConformer().GetAtomPosition(i) for i in range(ref_mol.GetNumAtoms())])
    center = ref_pos.mean(axis=0).tolist()
    size   = (ref_pos.max(axis=0) - ref_pos.min(axis=0) + 2 * pad_ref).clip(min=15).tolist()

if box_mode == "A" or center is None:
    use = np.array(ca_coords) if ca_coords else coords_np
    center = use.mean(axis=0).tolist()
    size   = (coords_np.max(axis=0) - coords_np.min(axis=0) + 2 * pad_auto).tolist()

print(f"  Box centre: ({center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f})")
print(f"  Box size  : ({size[0]:.3f}, {size[1]:.3f}, {size[2]:.3f})")

# 3c. convert receptor to PDBQT with charges
def _parse_pdbqt_charges(pdbqt_path):
    charges = []
    with open(pdbqt_path) as fh:
        for line in fh:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            q = None
            try:
                q = float(line[70:76].strip())
            except Exception:
                toks = line.split()
                for tok in reversed(toks[:-1]):
                    try:
                        q = float(tok)
                        break
                    except Exception:
                        pass
            if q is not None:
                charges.append(q)
    return charges

obC = ob.OBConversion()
obC.SetInAndOutFormats("pdb", "pdbqt")
obC.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid receptor

prot_mol = ob.OBMol()
if not obC.ReadFile(prot_mol, str(prot_in)):
    raise RuntimeError("OpenBabel could not read holo receptor PDB.")

cm = ob.OBChargeModel.FindType("gasteiger")
if cm:
    cm.ComputeCharges(prot_mol)

prot_pdbqt = WORKDIR / "protein_with_nadph.pdbqt"
obC.WriteFile(prot_mol, str(prot_pdbqt))

rec_charges = _parse_pdbqt_charges(prot_pdbqt)
if len(rec_charges) == 0:
    raise RuntimeError("No receptor charges could be parsed from PDBQT.")
if np.allclose(rec_charges, 0.0):
    raise RuntimeError("All receptor PDBQT charges are zero. Stop and inspect receptor preparation.")

print(f"  Receptor PDBQT -> {prot_pdbqt}")
print(f"  Receptor charge stats: n={len(rec_charges)}, min={np.min(rec_charges):.4f}, "
      f"max={np.max(rec_charges):.4f}, sum={np.sum(rec_charges):.4f}")


# ---------------------------------------------------------------------------
#  STEP 4 : Ligand parsing
# ---------------------------------------------------------------------------

_print_header("STEP 4/7", "Ligand Parsing")

mol_list = []
for idx, mol in enumerate(Chem.SDMolSupplier(str(lig_in), removeHs=False, sanitize=True)):
    if mol is None:
        continue
    raw = mol.GetPropsAsDict().get("_Name", "").strip()
    name = re.sub(r"[^\w\-]", "_", raw) if raw else f"LIG_{idx+1:04d}"
    mol_list.append((name, mol))

if ligand_limit is not None:
    mol_list = mol_list[:ligand_limit]

print(f"  Ligands ready for docking: {len(mol_list)}")
if len(mol_list) == 0:
    raise RuntimeError("No ligands parsed from prepared ligand SDF.")


# ---------------------------------------------------------------------------
#  STEP 5 : Docking loop
# ---------------------------------------------------------------------------

_print_header("STEP 5/7", f"Docking ({len(mol_list)} ligands)")

from vina import Vina

def _ob_sdf_to_pdbqt(sdf_path, pdbqt_path):
    obc = ob.OBConversion()
    obc.SetInAndOutFormats("sdf", "pdbqt")
    mol = ob.OBMol()
    if not obc.ReadFile(mol, str(sdf_path)):
        raise RuntimeError(f"OpenBabel could not read {sdf_path.name}")
    cm2 = ob.OBChargeModel.FindType("gasteiger")
    if cm2:
        cm2.ComputeCharges(mol)
    obc.WriteFile(mol, str(pdbqt_path))

def _split_pdbqt_models(pdbqt_path):
    models = []
    current = []
    with open(pdbqt_path) as fh:
        for line in fh:
            if line.startswith("MODEL"):
                current = []
            elif line.startswith("ENDMDL"):
                if current:
                    models.append(current)
                    current = []
            elif line.startswith(("ATOM", "HETATM")):
                current.append(line)
        if current and not models:
            models.append(current)
    return models

def _rdkit_from_pdbqt_models(template_mol, pdbqt_path, energies, lig_name):
    models = _split_pdbqt_models(pdbqt_path)
    out = []
    nat_template = template_mol.GetNumAtoms()

    for i, atom_lines in enumerate(models):
        if len(atom_lines) != nat_template:
            # atom-count mismatch => skip this pose
            continue
        mol_copy = copy.deepcopy(template_mol)
        conf = Chem.Conformer(nat_template)
        for aidx, line in enumerate(atom_lines):
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            conf.SetAtomPosition(aidx, Point3D(x, y, z))
        mol_copy.RemoveAllConformers()
        mol_copy.AddConformer(conf, assignId=True)
        mol_copy.SetProp("_Name", f"{lig_name}_Pose_{i+1}")
        if i < len(energies):
            mol_copy.SetProp("Vina_Affinity_kcal_mol", f"{energies[i][0]:.4f}")
            mol_copy.SetProp("RMSD_lower_bound_A", f"{energies[i][1]:.4f}")
            mol_copy.SetProp("RMSD_upper_bound_A", f"{energies[i][2]:.4f}")
            mol_copy.SetProp("Rank", str(i+1))
        out.append(mol_copy)
    return out

summary_rows = []
charge_rows = []
t0_total = time.time()

for lig_idx, (lig_name, mol_raw) in enumerate(mol_list, start=1):
    print(f"\n  {'-'*60}")
    print(f"  Ligand {lig_idx}/{len(mol_list)} | {lig_name}")

    try:
        mw      = Descriptors.ExactMolWt(mol_raw)
        logp    = Descriptors.MolLogP(mol_raw)
        hbd     = rdMolDescriptors.CalcNumHBD(mol_raw)
        hba     = rdMolDescriptors.CalcNumHBA(mol_raw)
        rotb    = rdMolDescriptors.CalcNumRotatableBonds(mol_raw)
        tpsa    = Descriptors.TPSA(mol_raw)
        formula = rdMolDescriptors.CalcMolFormula(mol_raw)

        if mol_raw.GetNumConformers() == 0 or not mol_raw.GetConformer().Is3D():
            raise RuntimeError("No 3D coordinates in input ligand.")

        lig_dir = WORKDIR / f"ligand_{lig_idx:04d}_{lig_name}"
        lig_dir.mkdir(exist_ok=True)

        lig_sdf = lig_dir / "ligand.sdf"
        lig_pdbqt = lig_dir / "ligand.pdbqt"

        w = Chem.SDWriter(str(lig_sdf))
        w.write(mol_raw)
        w.close()

        _ob_sdf_to_pdbqt(lig_sdf, lig_pdbqt)

        lig_charges = _parse_pdbqt_charges(lig_pdbqt)
        if len(lig_charges) == 0:
            raise RuntimeError("No ligand charges could be parsed from PDBQT.")
        if np.allclose(lig_charges, 0.0):
            raise RuntimeError("All ligand PDBQT charges are zero.")

        charge_rows.append({
            "Ligand_name": lig_name,
            "Charge_sum": round(float(np.sum(lig_charges)), 6),
            "Charge_min": round(float(np.min(lig_charges)), 6),
            "Charge_max": round(float(np.max(lig_charges)), 6),
            "N_atoms_in_pdbqt": len(lig_charges),
            "Status": "OK",
        })

        v = Vina(sf_name=scoring_fn,
                 cpu=cpu_cores if cpu_cores > 0 else 0,
                 seed=seed + lig_idx,
                 verbosity=0)
        v.set_receptor(str(prot_pdbqt))
        v.set_ligand_from_file(str(lig_pdbqt))
        v.compute_vina_maps(center=center, box_size=size)

        t0 = time.time()
        v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
        elapsed = time.time() - t0

        energies = v.energies(n_poses=n_poses)
        best_aff = float(energies[0][0])

        out_pdbqt = lig_dir / "docked_poses.pdbqt"
        v.write_poses(str(out_pdbqt), n_poses=n_poses, overwrite=True)

        # reconstruct SDF directly from original RDKit molecule to preserve bond orders/formal charges
        pose_mols = _rdkit_from_pdbqt_models(mol_raw, out_pdbqt, energies, lig_name)
        out_sdf = lig_dir / "docked_poses.sdf"
        wrt = Chem.SDWriter(str(out_sdf))
        for pm in pose_mols:
            wrt.write(pm)
        wrt.close()

        summary_rows.append({
            "Rank_overall": None,
            "Ligand_name": lig_name,
            "Formula": formula,
            "MW_Da": round(mw, 3),
            "LogP": round(logp, 3),
            "HBD": hbd,
            "HBA": hba,
            "RotB": rotb,
            "TPSA_A2": round(tpsa, 2),
            "Best_affinity_kcal_mol": round(best_aff, 4),
            "Best_RMSD_lb_A": round(float(energies[0][1]), 4),
            "Best_RMSD_ub_A": round(float(energies[0][2]), 4),
            "N_poses": len(energies),
            "Runtime_s": round(elapsed, 2),
            "Best_pose_sdf": str(out_sdf),
            "Status": "SUCCESS",
        })

        print(f"    Best affinity : {best_aff:.4f} kcal/mol")
        print(f"    Ligand charge : sum={np.sum(lig_charges):.4f}, min={np.min(lig_charges):.4f}, max={np.max(lig_charges):.4f}")
        print(f"    Docked poses  : {out_sdf}")

    except Exception as err:
        print(f"    FAILED: {err}")
        summary_rows.append({
            "Rank_overall": None,
            "Ligand_name": lig_name,
            "Formula": "--",
            "MW_Da": "--",
            "LogP": "--",
            "HBD": "--",
            "HBA": "--",
            "RotB": "--",
            "TPSA_A2": "--",
            "Best_affinity_kcal_mol": "--",
            "Best_RMSD_lb_A": "--",
            "Best_RMSD_ub_A": "--",
            "N_poses": 0,
            "Runtime_s": "--",
            "Best_pose_sdf": "--",
            "Status": f"FAILED: {err}",
        })
        charge_rows.append({
            "Ligand_name": lig_name,
            "Charge_sum": "",
            "Charge_min": "",
            "Charge_max": "",
            "N_atoms_in_pdbqt": "",
            "Status": f"FAILED: {err}",
        })
        if not skip_failed:
            raise

total_elapsed = time.time() - t0_total
print(f"\n  Total docking time: {total_elapsed:.1f} s")


# ---------------------------------------------------------------------------
#  STEP 6 : Ranking + top-10 final SDF
# ---------------------------------------------------------------------------

_print_header("STEP 6/7", "Ranking + Top Final Docked Pose SDF")

successful = [r for r in summary_rows if r["Status"] == "SUCCESS"]
successful.sort(key=lambda r: r["Best_affinity_kcal_mol"])
for rank, row in enumerate(successful, start=1):
    row["Rank_overall"] = rank

if affinity_cutoff is not None:
    successful = [r for r in successful if r["Best_affinity_kcal_mol"] <= affinity_cutoff]

top_final = successful[:final_top_n]
print(f"  Successful ligands : {len([r for r in summary_rows if r['Status'] == 'SUCCESS'])}")
print(f"  Final kept for merged SDF : {len(top_final)}")

top_sdf_path = WORKDIR / f"top_{final_top_n}_final_docked_poses.sdf"
w_top = Chem.SDWriter(str(top_sdf_path))
n_written = 0

for row in top_final:
    lig_name = row["Ligand_name"]
    lig_dirs = list(WORKDIR.glob(f"ligand_*_{lig_name}"))
    if not lig_dirs:
        continue
    pose_sdf = lig_dirs[0] / "docked_poses.sdf"
    if not pose_sdf.exists():
        continue

    mols = [m for m in Chem.SDMolSupplier(str(pose_sdf), removeHs=False) if m is not None]
    if not mols:
        continue

    best_pose = mols[0]  # rank-1 pose for that ligand
    best_pose.SetProp("Overall_Rank", str(row["Rank_overall"]))
    best_pose.SetProp("Ligand_Name", lig_name)
    best_pose.SetProp("Best_Affinity_kcal_mol", str(row["Best_affinity_kcal_mol"]))
    w_top.write(best_pose)
    n_written += 1

w_top.close()
print(f"  Merged top-pose SDF -> {top_sdf_path} ({n_written} molecules)")


# ---------------------------------------------------------------------------
#  STEP 7 : Outputs
# ---------------------------------------------------------------------------

_print_header("STEP 7/7", "Writing Reports and ZIP")

summary_csv = WORKDIR / "summary_results.csv"
_fields = [
    "Rank_overall", "Ligand_name", "Formula", "MW_Da", "LogP", "HBD", "HBA",
    "RotB", "TPSA_A2", "Best_affinity_kcal_mol", "Best_RMSD_lb_A", "Best_RMSD_ub_A",
    "N_poses", "Runtime_s", "Best_pose_sdf", "Status",
]
with open(summary_csv, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=_fields)
    writer.writeheader()
    writer.writerows(successful + [r for r in summary_rows if r["Status"] != "SUCCESS"])

charge_csv = WORKDIR / "charge_validation.csv"
with open(charge_csv, "w", newline="") as fh:
    fieldnames = ["Ligand_name", "Charge_sum", "Charge_min", "Charge_max", "N_atoms_in_pdbqt", "Status"]
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow({
        "Ligand_name": "RECEPTOR_WITH_NADPH",
        "Charge_sum": round(float(np.sum(rec_charges)), 6),
        "Charge_min": round(float(np.min(rec_charges)), 6),
        "Charge_max": round(float(np.max(rec_charges)), 6),
        "N_atoms_in_pdbqt": len(rec_charges),
        "Status": "OK",
    })
    writer.writerows(charge_rows)

config_txt = WORKDIR / "vina_box_config.txt"
with open(config_txt, "w") as fh:
    fh.write(f"receptor = {prot_pdbqt.name}\n")
    fh.write(f"center_x = {center[0]:.4f}\n")
    fh.write(f"center_y = {center[1]:.4f}\n")
    fh.write(f"center_z = {center[2]:.4f}\n")
    fh.write(f"size_x = {size[0]:.4f}\n")
    fh.write(f"size_y = {size[1]:.4f}\n")
    fh.write(f"size_z = {size[2]:.4f}\n")
    fh.write(f"exhaustiveness = {exhaustiveness}\n")
    fh.write(f"num_modes = {n_poses}\n")
    fh.write(f"seed = {seed}\n")
    fh.write(f"scoring = {scoring_fn}\n")

out_zip = Path(f"/content/{run_name}_results.zip")
zip_top = f"{run_name}_results"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in [summary_csv, charge_csv, config_txt, top_sdf_path, prot_pdbqt]:
        if f.exists():
            zf.write(f, f"{zip_top}/{f.name}")
    for ld in sorted(WORKDIR.glob("ligand_*")):
        if ld.is_dir():
            for fp in sorted(ld.rglob("*")):
                if fp.is_file():
                    zf.write(fp, f"{zip_top}/{fp.relative_to(WORKDIR)}")

print(f"  Summary CSV          -> {summary_csv}")
print(f"  Charge validation    -> {charge_csv}")
print(f"  Final merged top SDF -> {top_sdf_path}")
print(f"  ZIP                  -> {out_zip}")

print("\n" + "=" * 72)
print("  PIPELINE COMPLETE")
print("=" * 72)
print(f"  Receptor used     : {prot_in}")
print(f"  Ligand SDF used   : {lig_in}")
print(f"  Receptor PDBQT    : {prot_pdbqt}")
print(f"  Top merged SDF    : {top_sdf_path}")
print(f"  Successful docks  : {len([r for r in summary_rows if r['Status'] == 'SUCCESS'])}")
print(f"  Total runtime     : {total_elapsed:.1f} s")
print("=" * 72)

from google.colab import files as colab_files
colab_files.download(str(summary_csv))
colab_files.download(str(charge_csv))
colab_files.download(str(top_sdf_path))
colab_files.download(str(out_zip))